# 04. Метрики классификации

## Только классификация

Accuracy, Precision, Recall, F1, ROC AUC и PR AUC отвечают на разные вопросы о качестве **классификатора**.

Начнём с простой идеи:

> Насколько часто модель угадала класс?

Потом увидим, почему одной Accuracy недостаточно, особенно при **несбалансированных классах**, и перейдём к матрице ошибок, Precision, Recall и F1.

После этого отдельно разберём ROC AUC и Precision-Recall Curve.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, auc
)

sns.set_theme(style="whitegrid")

## 1. Матрица ошибок — основа метрик

Для бинарной классификации введём четыре типа результата:

- **True Positive (TP)** — объект класса 1 правильно отнесён к классу 1.
- **True Negative (TN)** — объект класса 0 правильно отнесён к классу 0.
- **False Positive (FP)** — объект класса 0 ошибочно отнесён к классу 1.
- **False Negative (FN)** — объект класса 1 ошибочно отнесён к классу 0.

Именно из этих четырёх чисел строится большинство классических метрик классификации.

## 2. Accuracy

$$
Accuracy=\frac{TP+TN}{TP+TN+FP+FN}.
$$

Accuracy отвечает на вопрос:

> Какую долю всех объектов модель классифицировала правильно?

Проблема появляется при **дисбалансе классов**.

### Пример

Допустим, 1000 объектов:

- 990 объектов относятся к классу 0;
- 10 объектов — к классу 1.

Если модель всегда говорит «класс 0», то:

$$
Accuracy=\frac{990}{1000}=99\%.
$$

На первый взгляд результат выглядит великолепно. Но модель **не нашла ни одного объекта редкого класса**.

Поэтому главный недостаток Accuracy:

> При сильном дисбалансе классов высокая Accuracy может скрывать полностью бесполезный классификатор для редкого класса.

In [ ]:
# Иллюстрация несбалансированной классификации
y_true = np.array([0]*990 + [1]*10)
y_pred = np.zeros_like(y_true)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))

## 3. Precision

$$
Precision=\frac{TP}{TP+FP}.
$$

Precision отвечает:

> Среди объектов, которым модель присвоила положительный класс, какая доля действительно является положительной?

Высокий Precision означает мало **ложных положительных** результатов.

### Недостаток Precision

Precision не показывает, сколько положительных объектов модель **пропустила**. Можно получить высокий Precision, предсказывая положительный класс только в нескольких очень уверенных случаях.

## 4. Recall

$$
Recall=\frac{TP}{TP+FN}.
$$

Recall отвечает:

> Какую долю всех настоящих положительных объектов модель смогла обнаружить?

Высокий Recall означает мало **ложных отрицательных** результатов.

### Недостаток Recall

Можно добиться высокого Recall, если очень часто объявлять объект положительным. Тогда число False Positive может сильно вырасти и Precision снизится.

## 5. F1-score

F1 — гармоническое среднее Precision и Recall:

$$
F1=2\cdot\frac{Precision\cdot Recall}{Precision+Recall}.
$$

F1 полезна, когда нам одновременно важны Precision и Recall.

### Недостатки F1

- Не учитывает TN.
- Одним числом скрывает компромисс между двумя метриками.
- Не всегда подходит, если цена FP и FN существенно различается.
- F1 не оценивает качество самих вероятностей — две модели могут иметь одинаковый F1, но очень разные вероятностные прогнозы.

In [ ]:
# Небольшой искусственный пример
y_true = np.array([1,1,1,1,0,0,0,0,0,0])
y_pred = np.array([1,1,0,1,0,0,1,0,0,0])

print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

# 6. Confusion Matrix

Confusion matrix показывает, **какие именно ошибки** делает классификатор.

В бинарной задаче она имеет вид:

| | Предсказан 0 | Предсказан 1 |
|---|---:|---:|
| Истинный 0 | TN | FP |
| Истинный 1 | FN | TP |

Это особенно полезно, когда важно понимать характер ошибок, а не только получить одно число.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm, annot=True, fmt="d", cbar=False,
    xticklabels=["Предсказан 0", "Предсказан 1"],
    yticklabels=["Истинный 0", "Истинный 1"]
)
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.title("Confusion Matrix")
plt.show()

# 7. Откуда берётся ROC Curve?

Многие классификаторы сначала выдают **вероятность** принадлежности к положительному классу.

Например:

| Объект | P(y=1) |
|---|---:|
| A | 0.95 |
| B | 0.80 |
| C | 0.60 |
| D | 0.40 |
| E | 0.20 |

Чтобы получить класс, выбираем порог.

При пороге `0.5`:

- `0.95`, `0.80`, `0.60` → класс 1;
- `0.40`, `0.20` → класс 0.

Но порог можно менять. При каждом пороге изменяются TP, FP, TN и FN.

Для ROC используются:

$$
TPR=\frac{TP}{TP+FN}
$$

и

$$
FPR=\frac{FP}{FP+TN}.
$$

`TPR` — это Recall.

`FPR` показывает долю отрицательных объектов, ошибочно признанных положительными.

## 8. ROC Curve на простом примере

Построим классификатор и получим вероятности. Затем будем менять порог от высокого к низкому.

Для каждого порога вычислим одну точку:

\[
(FPR,TPR).
\]

Соединив точки, получаем **ROC-кривую**.

In [ ]:
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    class_sep=1.2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, proba)
roc_auc = roc_auc_score(y_test, proba)

print("ROC AUC:", roc_auc)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Случайный классификатор")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curve")
plt.legend()
plt.show()

### Как читать ROC-график

- Ось X — `FPR`: сколько отрицательных объектов мы ошибочно объявили положительными.
- Ось Y — `TPR`: сколько настоящих положительных объектов мы смогли обнаружить.
- Левая верхняя область графика желательна.
- Диагональ соответствует примерно случайному ранжированию.

**ROC AUC** — площадь под ROC-кривой.

Интуитивно AUC показывает, насколько хорошо модель **разделяет и ранжирует** положительные и отрицательные объекты по своим оценкам.

Приблизительно:

- `AUC = 0.5` — уровень случайного ранжирования;
- `AUC → 1` — очень хорошее разделение;
- `AUC < 0.5` — модель ранжирует хуже случайного, хотя инверсия её оценок могла бы дать результат лучше.

## 9. Важное отличие ROC AUC от Accuracy

Accuracy требует выбрать конкретный порог и после этого считает правильные ответы.

ROC AUC рассматривает **множество порогов** и оценивает способность модели разделять классы.

Но у ROC AUC тоже есть ограничение: при сильном дисбалансе классов FPR может выглядеть небольшим даже тогда, когда число ложных положительных результатов является практически значимым.

Поэтому для редкого положительного класса часто полезно дополнительно смотреть **Precision-Recall Curve**.

# 10. Precision-Recall Curve

Для каждого порога можно вычислять:

\[
Precision=\frac{TP}{TP+FP}
\]

и

\[
Recall=\frac{TP}{TP+FN}.
\]

Получаем кривую:

- X — Recall;
- Y — Precision.

Площадь под ней называют **PR AUC** (часто также используется Average Precision как связанная с PR-кривой метрика).

PR-кривая особенно информативна, когда положительный класс редкий и нас интересует качество именно положительных предсказаний.

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8,6))
plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

print("PR AUC:", pr_auc)

## 11. Как выбирать метрику?

### Accuracy
Используйте, если классы относительно сбалансированы и ошибки примерно одинаково важны.

### Precision
Важен, когда **ложное положительное решение дорого**.

### Recall
Важен, когда **пропуск положительного объекта особенно опасен**.

### F1
Удобен, когда нужен баланс между Precision и Recall и TN не является центральной частью задачи.

### ROC AUC
Полезен для оценки способности модели различать классы по вероятностным/скоринговым оценкам при разных порогах.

### PR AUC
Особенно полезен при **редком положительном классе**, когда важно одновременно сохранять Precision и Recall.

### Confusion Matrix
Стоит смотреть практически всегда, если необходимо понимать структуру ошибок.

**Главный вывод:** не существует одной универсально лучшей метрики. Метрика должна соответствовать тому, **какие ошибки классификатора являются наиболее важными для конкретной задачи**.